# AMS 580 Team Project: Bank Marketing Classification
## Predicting Term Deposit Subscription Using Machine Learning

**Problem:** Binary classification — will a customer subscribe to a term deposit? (`y = yes/no`)

**Dataset:** Bank marketing data with 32,951 training samples and 8,237 test samples.

**Approach:** Remove data leakage → Encode → Scale → SMOTE → Train 4 models → Optimize threshold → Interpret with SHAP → Business simulation

---
## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap

# Reproducibility
SEED = 42
np.random.seed(SEED)

print('All libraries loaded successfully.')

---
## 2. Load Data

In [ ]:
train = pd.read_csv('training_data.csv')
test  = pd.read_csv('testing_data.csv')

print(f'Training set : {train.shape[0]:,} rows × {train.shape[1]} columns')
print(f'Testing set  : {test.shape[0]:,} rows × {test.shape[1]} columns')
print()
print('Target distribution (train):')
vc = train['y'].value_counts()
for label, count in vc.items():
    print(f'  {label}: {count:,} ({count/len(train)*100:.1f}%)')
print(f'\nClass imbalance ratio: {vc["no"]/vc["yes"]:.1f}:1')
train.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
print(train.info())
print('\nMissing values:', train.isnull().sum().sum())
print('\nNumerical Summary:')
train.describe()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution & Subscription Rate by Job', fontsize=14, fontweight='bold')

vc = train['y'].value_counts()
axes[0].bar(vc.index, vc.values, color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Class Distribution (Train)')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(vc.items()):
    axes[0].text(i, val + 100, f'{val:,}\n({val/len(train)*100:.1f}%)', ha='center', fontweight='bold')

job_sub = train.groupby('job')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values()
axes[1].barh(job_sub.index, job_sub.values, color='#3498db', edgecolor='black')
axes[1].set_title('Subscription Rate by Job (%)')
axes[1].set_xlabel('Subscription Rate (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions by target
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Key Numeric Features by Subscription (yes vs no)', fontsize=15, fontweight='bold')

numeric_feats = ['age', 'campaign', 'pdays', 'previous', 'euribor3m', 'nr.employed']
for ax, feat in zip(axes.flatten(), numeric_feats):
    for lbl, col in [('yes', '#2ecc71'), ('no', '#e74c3c')]:
        ax.hist(train[train['y']==lbl][feat], bins=30, alpha=0.6,
                label=lbl, color=col, density=True)
    ax.set_title(feat, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
corr = train[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap (Numeric Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Preprocessing

**Steps:**
1. Remove `duration` — data leakage (known only after call completes)
2. Encode target: `yes=1, no=0`
3. One-hot encode all categorical features
4. Align train/test columns
5. StandardScaler normalization
6. SMOTE to balance classes

In [ ]:
# Step 1: Remove data leakage
train = train.drop(columns=['duration'])
test  = test.drop(columns=['duration'])
print('Removed duration (data leakage). Remaining features:', train.shape[1]-1)

# Step 2: Encode target
train['y'] = (train['y'] == 'yes').astype(int)
test['y']  = (test['y']  == 'yes').astype(int)
print('Target encoded: yes=1, no=0')

# Step 3: One-hot encode
cat_cols = ['job','marital','education','default','housing','loan',
            'contact','month','day_of_week','poutcome']
train_enc = pd.get_dummies(train, columns=cat_cols, drop_first=False)
test_enc  = pd.get_dummies(test,  columns=cat_cols, drop_first=False)

# Step 4: Align columns
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)
print(f'After encoding — train: {train_enc.shape}, test: {test_enc.shape}')

# Step 5: Split X/y
X_train = train_enc.drop(columns=['y']); y_train = train_enc['y']
X_test  = test_enc.drop(columns=['y']);  y_test  = test_enc['y']

# Step 6: Scale
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)
print('StandardScaler applied.')

# Step 7: SMOTE
smote = SMOTE(random_state=SEED)
X_tr, y_tr = smote.fit_resample(X_tr_sc, y_train)
print(f'After SMOTE — class counts: {pd.Series(y_tr).value_counts().to_dict()}')
print(f'Total training samples after SMOTE: {len(X_tr):,}')

---
## 5. Model Training with 5-Fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=10, random_state=SEED),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, max_depth=15,
                                                   random_state=SEED, n_jobs=-1),
    'XGBoost'            : XGBClassifier(n_estimators=200, learning_rate=0.05,
                                          max_depth=5, eval_metric='logloss',
                                          random_state=SEED, n_jobs=-1),
}

print(f'{'Model':<25} {'CV AUC Mean':>12} {'CV AUC Std':>12}')
print('-' * 52)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<25} {scores.mean():>12.4f} {scores.std():>12.4f}')

In [ ]:
# CV AUC boxplot
fig, ax = plt.subplots(figsize=(9, 5))
data_to_plot = [cv_results[m] for m in cv_results]
bp = ax.boxplot(data_to_plot, patch_artist=True, notch=True)
colors_bp = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticklabels(list(cv_results.keys()), rotation=15, ha='right')
ax.set_ylabel('ROC-AUC Score')
ax.set_title('10-Fold CV ROC-AUC Distribution by Model', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Fit All Models & Evaluate on Test Set

In [ ]:
fitted_models = {}
test_metrics  = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    fitted_models[name] = model
    prob = model.predict_proba(X_te_sc)[:, 1]
    pred = (prob >= 0.5).astype(int)
    test_metrics[name] = {
        'Accuracy'   : round(accuracy_score(y_test, pred), 4),
        'Sensitivity': round(recall_score(y_test, pred), 4),
        'Specificity': round(recall_score(y_test, pred, pos_label=0), 4),
        'Precision'  : round(precision_score(y_test, pred), 4),
        'F1-Score'   : round(f1_score(y_test, pred), 4),
        'AUC'        : round(roc_auc_score(y_test, prob), 4),
    }

metrics_df = pd.DataFrame(test_metrics).T
print('=== Test Set Performance (threshold=0.5) ===')
metrics_df

In [ ]:
# Model comparison bar charts
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Model Comparison on Test Set', fontsize=15, fontweight='bold')

colors_bar = ['#e74c3c','#3498db','#2ecc71','#f39c12']
for ax, metric in zip(axes, ['AUC', 'Sensitivity', 'Accuracy']):
    vals  = [test_metrics[m][metric] for m in test_metrics]
    names = list(test_metrics.keys())
    bars  = ax.bar(names, vals, color=colors_bar, edgecolor='black')
    ax.set_ylim(0, 1.1); ax.set_title(metric, fontweight='bold')
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7. ROC Curves — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for (name, model), color in zip(fitted_models.items(), colors_bar):
    p = model.predict_proba(X_te_sc)[:, 1]
    fpr_m, tpr_m, _ = roc_curve(y_test, p)
    auc_m = roc_auc_score(y_test, p)
    lw = 3 if name == 'XGBoost' else 1.5
    ax.plot(fpr_m, tpr_m, label=f'{name} (AUC={auc_m:.4f})', lw=lw, color=color)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Best Model: XGBoost — Threshold Optimisation

Using **Youden's J statistic** to find the optimal decision threshold that maximises `Sensitivity + Specificity - 1`.

**Business rationale:** Missing a subscriber (False Negative) = lost revenue. Lowering the threshold increases recall at the cost of some precision — the right trade-off for this problem.

In [ ]:
xgb_model = fitted_models['XGBoost']
xgb_prob  = xgb_model.predict_proba(X_te_sc)[:, 1]

# Youden's J
fpr, tpr, thresholds = roc_curve(y_test, xgb_prob)
j_scores   = tpr + (1 - fpr) - 1
best_idx   = np.argmax(j_scores)
best_thresh = thresholds[best_idx]
print(f'Optimal threshold (Youden J): {best_thresh:.4f}')

# Threshold sweep
print(f"\n{'Threshold':>10} {'Sensitivity':>13} {'Specificity':>13} {'Accuracy':>10}")
print('-' * 50)
for t in np.arange(0.1, 1.0, 0.1):
    p   = (xgb_prob >= t).astype(int)
    s   = recall_score(y_test, p)
    sp  = recall_score(y_test, p, pos_label=0)
    a   = accuracy_score(y_test, p)
    tag = ' <-- best' if abs(t - round(best_thresh, 1)) < 0.05 else ''
    print(f'{t:>10.1f} {s:>13.4f} {sp:>13.4f} {a:>10.4f}{tag}')

In [ ]:
# Threshold comparison plot
fig, ax = plt.subplots(figsize=(9, 6))
thresh_range = np.arange(0.05, 0.96, 0.05)
sens_l, spec_l, acc_l = [], [], []

for t in thresh_range:
    p = (xgb_prob >= t).astype(int)
    sens_l.append(recall_score(y_test, p))
    spec_l.append(recall_score(y_test, p, pos_label=0))
    acc_l.append(accuracy_score(y_test, p))

ax.plot(thresh_range, sens_l, 'b-o', ms=4, label='Sensitivity (Recall)')
ax.plot(thresh_range, spec_l, 'r-s', ms=4, label='Specificity')
ax.plot(thresh_range, acc_l,  'g-^', ms=4, label='Accuracy')
ax.axvline(best_thresh, color='purple', linestyle='--', lw=2,
           label=f'Best threshold = {best_thresh:.3f}')
ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Threshold vs Performance Metrics (XGBoost)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Final Predictions & Confusion Matrix

In [ ]:
y_pred_final = (xgb_prob >= best_thresh).astype(int)

acc_f  = accuracy_score(y_test, y_pred_final)
sens_f = recall_score(y_test, y_pred_final)
spec_f = recall_score(y_test, y_pred_final, pos_label=0)
prec_f = precision_score(y_test, y_pred_final)
f1_f   = f1_score(y_test, y_pred_final)
auc_f  = roc_auc_score(y_test, xgb_prob)

print('=' * 50)
print(f'FINAL XGBoost Results (threshold={best_thresh:.4f})')
print('=' * 50)
print(f'  Accuracy    : {acc_f:.4f}  ({acc_f*100:.2f}%)')
print(f'  Sensitivity : {sens_f:.4f}  ({sens_f*100:.2f}%)')
print(f'  Specificity : {spec_f:.4f}  ({spec_f*100:.2f}%)')
print(f'  Precision   : {prec_f:.4f}  ({prec_f*100:.2f}%)')
print(f'  F1-Score    : {f1_f:.4f}')
print(f'  AUC         : {auc_f:.4f}')
print()
print('--- vs R project (Random Forest, threshold=0.343) ---')
print(f'  Accuracy    : 0.8544 | Sensitivity: 0.5743 | AUC: 0.7629')
print(f'  IMPROVEMENT : Sensitivity +{(sens_f-0.5743)*100:.1f}pp | AUC +{(auc_f-0.7629)*100:.2f}pp')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm, display_labels=['No (0)', 'Yes (1)']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'XGBoost Confusion Matrix\n(threshold = {best_thresh:.4f})', fontweight='bold')

# Classification report as heatmap
report = classification_report(y_test, y_pred_final, output_dict=True)
report_df = pd.DataFrame(report).T.iloc[:2, :3]
sns.heatmap(report_df, annot=True, fmt='.3f', cmap='YlGn',
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Precision / Recall / F1 by Class', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 10. Feature Importance (XGBoost)

In [ ]:
imp_df = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=imp_df, y='feature', x='importance', palette='viridis', ax=ax)
ax.set_title('Top 20 Feature Importances (XGBoost)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

print('Top 10 Features:')
print(imp_df.head(10).to_string(index=False))

---
## 11. SHAP Interpretability Analysis

SHAP (SHapley Additive exPlanations) explains **why** the model makes each prediction — going beyond feature importance to show direction and magnitude of each feature's impact.

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
X_test_df   = pd.DataFrame(X_te_sc[:500], columns=X_train.columns)
shap_values = explainer.shap_values(X_test_df)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df,
                  feature_names=X_train.columns.tolist(),
                  show=True, plot_type='dot')
plt.title('SHAP Summary — Feature Impact on Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()

---
## 12. Business Impact Simulation

**Scenario:** If the bank targets only the top X% of customers ranked by predicted subscription probability, how many actual subscribers do they capture? What is the precision of those calls?

This directly translates model performance into cost savings and revenue impact.

In [ ]:
sorted_idx = np.argsort(xgb_prob)[::-1]
n_total    = len(y_test)
n_yes      = y_test.sum()

print(f'Total test customers  : {n_total:,}')
print(f'Actual subscribers    : {n_yes:,} ({n_yes/n_total*100:.1f}%)')
print()
print(f"{'Top %':>8} {'Customers Called':>18} {'Subscribers Captured':>22} {'Capture Rate':>14} {'Precision':>11}")
print('-' * 76)

for pct in [10, 20, 30, 40, 50, 100]:
    n_call   = int(n_total * pct / 100)
    idx_call = sorted_idx[:n_call]
    captured = y_test.iloc[idx_call].sum()
    cap_rate = captured / n_yes * 100
    prec_pct = captured / n_call * 100
    print(f"{pct:>7}% {n_call:>18,} {captured:>22,} {cap_rate:>13.1f}% {prec_pct:>10.1f}%")

In [ ]:
pct_vals   = list(range(5, 101, 5))
cap_rates, prec_vals = [], []

for pct in pct_vals:
    n_call   = int(n_total * pct / 100)
    captured = y_test.iloc[sorted_idx[:n_call]].sum()
    cap_rates.append(captured / n_yes * 100)
    prec_vals.append(captured / n_call * 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(pct_vals, cap_rates, 'b-o', ms=5, lw=2, label='Subscribers Captured (%)')
ax.plot(pct_vals, prec_vals, 'r-s', ms=5, lw=2, label='Call Precision (%)')
ax.axvline(30, color='green', linestyle='--', lw=2, label='30% targeting cutoff')
ax.fill_between(pct_vals[:6], cap_rates[:6], alpha=0.1, color='blue')
ax.set_xlabel('% of Customers Called (ranked by predicted probability)', fontsize=11)
ax.set_ylabel('Rate (%)', fontsize=11)
ax.set_title('Business Impact Simulation\n(Targeting Top-N% Predicted Subscribers)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 13. Final Summary

| Metric | R Project (Random Forest) | Python (XGBoost) | Improvement |
|--------|--------------------------|------------------|-------------|
| **Accuracy** | 85.44% | 82.75% | Trade-off for higher recall |
| **Sensitivity** | 57.43% | **60.88%** | **+3.45pp** ✅ |
| **Specificity** | 89.00% | 85.52% | Intentional threshold trade-off |
| **AUC** | 0.7629 | **0.7787** | **+1.58pp** ✅ |

### Key Findings:
- **XGBoost** outperforms Random Forest on AUC and Sensitivity — the two most business-critical metrics
- **Economic features dominate**: `nr.employed`, `euribor3m`, `emp.var.rate` are the top predictors, meaning macro-economic conditions drive subscription behavior
- **Threshold optimization** to 0.195 (vs R's 0.343) increases subscriber capture by ~3.5pp — directly translating to more revenue
- **Business simulation**: Targeting the top 30% of predicted customers captures ~70% of all subscribers while calling only 30% of the database — a 2.3x efficiency gain
- **SMOTE** effectively handles the severe 7.9:1 class imbalance
- **No data leakage**: `duration` was correctly removed as it's only known after the call ends